In [5]:
import math


def dimensionar_celula_capacitiva(
    potencia_total_kvar,
    tensao_total_kv,
    frequencia_hz=60.0,
    tensao_alvo_elemento_kv=2.0,
    potencia_alvo_elemento_kvar=12.0,
):
    """
    Determines the optimal number of capacitor elements in series and parallel.

    Parameters
    ----------
    potencia_total_kvar : float
        Total reactive power of the capacitor cell in kVAr.

    tensao_total_kv : float
        Total voltage of the capacitor cell in kV.

    frequencia_hz : float, optional
        System frequency in Hz. Default is 60 Hz.

    tensao_alvo_elemento_kv : float, optional
        Desired voltage per capacitor element in kV.
        Default is 2 kV.

    potencia_alvo_elemento_kvar : float, optional
        Desired reactive power per capacitor element in kVAr.
        Default is 8 kVAr.

    Returns
    -------
    dict
        Dictionary containing:
        - Ns: number of elements in series
        - Np: number of elements in parallel
        - total_elements: total number of elements
        - voltage_element_kv: voltage per element
        - reactive_power_element_kvar: reactive power per element
        - capacitance_element_uf: capacitance per element
        - voltage_error_percent: voltage error
        - reactive_power_error_percent: reactive power error
        - optimization_score: combined optimization error
    """

    # Validate input data
    if potencia_total_kvar <= 0:
        raise ValueError("potencia_total_kvar must be greater than zero.")

    if tensao_total_kv <= 0:
        raise ValueError("tensao_total_kv must be greater than zero.")

    if frequencia_hz is None:
        frequencia_hz = 60.0

    if frequencia_hz <= 0:
        raise ValueError("frequencia_hz must be greater than zero.")

    # Estimate a reasonable search limit for the number of series elements
    ns_estimado = tensao_total_kv / tensao_alvo_elemento_kv

    ns_max = max(
        50,
        math.ceil(4 * ns_estimado + 20),
    )

    melhor_resultado = None

    for ns in range(1, ns_max + 1):

        # Voltage applied to each capacitor element
        tensao_elemento_kv = tensao_total_kv / ns

        # Estimate the ideal number of parallel branches
        np_ideal = potencia_total_kvar / (
            potencia_alvo_elemento_kvar * ns
        )

        np_base = max(1, math.floor(np_ideal))

        # Only nearby integer values need to be evaluated
        candidatos_np = {
            1,
            np_base,
            np_base + 1,
            max(1, np_base - 1),
        }

        for np in candidatos_np:

            # Reactive power handled by each individual element
            potencia_elemento_kvar = (
                potencia_total_kvar / (ns * np)
            )

            # Relative voltage error
            erro_tensao = (
                tensao_elemento_kv
                - tensao_alvo_elemento_kv
            ) / tensao_alvo_elemento_kv

            # Relative reactive power error
            erro_potencia = (
                potencia_elemento_kvar
                - potencia_alvo_elemento_kvar
            ) / potencia_alvo_elemento_kvar

            # Combined normalized error
            score = math.sqrt(
                erro_tensao**2
                + erro_potencia**2
            )

            # Capacitance calculation:
            # Q = 2*pi*f*C*V^2
            capacitancia_elemento_uf = (
                potencia_elemento_kvar * 1000
                / (
                    2
                    * math.pi
                    * frequencia_hz
                    * tensao_elemento_kv**2
                )
            )

            total_elementos = ns * np

            resultado = {
                "Ns": ns,
                "Np": np,
                "total_elements": total_elementos,
                "voltage_element_kv": tensao_elemento_kv,
                "reactive_power_element_kvar": potencia_elemento_kvar,
                "capacitance_element_uf": capacitancia_elemento_uf,
                "voltage_error_percent": erro_tensao * 100,
                "reactive_power_error_percent": erro_potencia * 100,
                "optimization_score": score,
            }

            # Select the smallest combined error
            if melhor_resultado is None:
                melhor_resultado = resultado
                continue

            score_atual = melhor_resultado["optimization_score"]

            if score < score_atual:
                melhor_resultado = resultado

            elif math.isclose(
                score,
                score_atual,
                rel_tol=1e-12,
                abs_tol=1e-12,
            ):
                # In case of a tie, prefer fewer capacitor elements
                if (
                    total_elementos
                    < melhor_resultado["total_elements"]
                ):
                    melhor_resultado = resultado

    return melhor_resultado


if __name__ == "__main__":

    # Example
    resultado = dimensionar_celula_capacitiva(
        potencia_total_kvar=662.7,
        tensao_total_kv=8.468,
        frequencia_hz=60,
    )

    print("Optimal capacitor cell configuration")
    print("------------------------------------")
    print(f"Series elements (Ns): {resultado['Ns']}")
    print(f"Parallel elements (Np): {resultado['Np']}")
    print(
        f"Total elements: "
        f"{resultado['total_elements']}"
    )
    print(
        f"Voltage per element: "
        f"{resultado['voltage_element_kv']:.4f} kV"
    )
    print(
        f"Reactive power per element: "
        f"{resultado['reactive_power_element_kvar']:.4f} kVAr"
    )
    print(
        f"Capacitance per element: "
        f"{resultado['capacitance_element_uf']:.4f} uF"
    )
    print(
        f"Voltage error: "
        f"{resultado['voltage_error_percent']:.2f} %"
    )
    print(
        f"Reactive power error: "
        f"{resultado['reactive_power_error_percent']:.2f} %"
    )

Optimal capacitor cell configuration
------------------------------------
Series elements (Ns): 4
Parallel elements (Np): 14
Total elements: 56
Voltage per element: 2.1170 kV
Reactive power per element: 11.8339 kVAr
Capacitance per element: 7.0042 uF
Voltage error: 5.85 %
Reactive power error: -1.38 %
